# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 7: Recommender Systems — Collaborative Filtering — ✅ Solution
**Estimated time: 1 hour**

### Learning Objectives
By the end of this exercise, you will be able to:
- Understand how **Collaborative Filtering** works for building recommender systems
- Load, clean, and merge **movie rating data**
- Create and analyze a **user-movie rating matrix**
- Find **similar movies** using **Pearson correlation**
- Visualize **rating distributions** and uncover patterns in user preferences

In this exercise, you will build a **collaborative filtering recommender system** using movie ratings data. You will clean the data, compute rating statistics, and find movies similar to *Star Wars* and *Liar Liar* based on user rating patterns.

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

---
## Setup

🏃 **RUN** the cell below to import libraries.

In [ ]:
# Import needed libraries
# Author: Luca Gudi (lgg.digi@cbs.dk)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 1. Load and Explore the Data

📖 **READ**: We will work with two datasets from the **MovieLens 100K** collection:

1. **`user_data.csv`** — Contains 100,000 movie ratings from users
   - Columns: `user_id`, `item_id`, `rating`, `timestamp`
   - Tab-separated, no header row

2. **`Movie_Id_Titles.csv`** — Maps each `item_id` to its movie title

The goal is to build a **collaborative filtering** system that finds movies similar to a given movie, based on how users rated them.

🏃 **RUN** the cells below to load both datasets.

In [ ]:
# Specify Column Names. They are not included in the file
column_names = ['user_id', 'item_id', 'rating', 'timestamp']

# Load the user data
user_data = pd.read_csv("user_data.csv", sep='\t', names=column_names)
# item_id is the key for the movie. Rating is the score user_id gave to the item_id

# Load the Movie Data
movie_metadata = pd.read_csv("Movie_Id_Titles.csv")

print(f"User data shape: {user_data.shape}")
print(f"Movie metadata shape: {movie_metadata.shape}")
print("\nUser data sample:")
user_data.head()

In [ ]:
movie_metadata.head()

---
## 2. Data Cleaning — Extract Publication Year

📖 **READ**: The movie titles contain the **publication year** in parentheses, e.g., `"Toy Story (1995)"`. We want to extract this into a separate column for analysis.

However, the data isn't perfectly clean:
- One title is just `"unknown"` — extracting the last 6 characters gives `"nknown"`
- One title has an unusual format `"(1995) (V)"` that produces the artifact `"5V"`

We'll clean these step by step.

✏️ **TODO**: Extract the publication year from the `title` column and store it in a new column called `"Publishment Year"`. Then clean up any anomalies.

*Hints:*
- *Extract the last 6 characters from the title with `str[-6:]`*
- *Remove parentheses: first `str.replace("(", "")`, then `str.replace(")", "")`*
- *Remove spaces with `str.replace(" ", "")`*
- *Replace `"nknown"` with `"1996"` (the most common year)*
- *Check `unique()` values to spot any remaining anomalies*
- *Use `str.contains("5V")` to find where the `"5V"` anomaly comes from*

In [ ]:
# Get the last six characters of Title
movie_metadata["Publishment Year"] = movie_metadata["title"].str[-6:]

# Remove Parentheses from string
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].str.replace("(", "")
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].str.replace(")", "")

# Remove spaces
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].str.replace(" ", "")

# Replace nknown with 1996 (most common year)
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].str.replace("nknown", "1996")

# Show all values
print(movie_metadata["Publishment Year"].unique())

# Try to find where the value 5V is coming from
movie_metadata[movie_metadata["Publishment Year"].str.contains("5V")]

✏️ **TODO**: Fix the remaining anomaly — the value `"5V"` comes from the title `"Land Before Time III: The Time of the Great Giving (1995) (V)"`. Replace `"5V"` with `"1995"`, then convert the column to integer type.

*Hints:*
- *Use `str.replace("5V", "1995")` to fix the anomalous entry*
- *Convert with `.astype(int)`*
- *Print `unique()` to verify all values are valid years*

In [ ]:
# This row did not match our pattern because of ((1995) (V)). Replace by original value (1995). See title column
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].str.replace("5V", "1995")
# Now, we can convert to Int type
movie_metadata["Publishment Year"] = movie_metadata["Publishment Year"].astype(int)
# Show all values to verify operations
print(movie_metadata["Publishment Year"].unique())

---
## 3. Merge Datasets

📖 **READ**: We have two separate DataFrames — one with user ratings and one with movie titles. To analyze ratings by movie name, we need to **merge** (join) them on the shared `item_id` column.

We use a **left join** so that all rows from `user_data` are kept, even if a movie title is missing.

✏️ **TODO**: Merge both DataFrames on `item_id` using a left join. Print the shapes before and after merging.

*Hints:*
- *Use `pd.merge(user_data, movie_metadata, on='item_id', how='left')`*
- *Print the shape of both input DataFrames and the merged result*

In [ ]:
# Print Dimensionality
print("User Data: ", user_data.shape, "Movies: ", movie_metadata.shape)
# Join on item_id, only keep on index
user_movie_data = pd.merge(user_data, movie_metadata, how="left", on='item_id')
print("Joined Data: ", user_movie_data.shape)
user_movie_data.head()

---
## 4. Compute Rating Statistics

📖 **READ**: Before building the recommender, let's understand the data. We'll compute two key metrics for each movie:

- **Number of Ratings** — How many users rated the movie
- **Mean Rating** — The average score the movie received

Movies with very few ratings might have unreliable averages (e.g., a movie rated 5.0 by only one person).

✏️ **TODO**: Create a `ratings` DataFrame with the number of ratings and mean rating for each movie.

*Hints:*
- *Group by `'title'` and use `.count()` on the `'rating'` column*
- *Rename the column using `.rename(columns={"rating": "Number of Ratings"})`*
- *Add a new column `'Mean Rating'` using `groupby('title')['rating'].mean()`*

In [ ]:
# Create an average rating and number of ratings column
ratings = pd.DataFrame(user_movie_data.groupby('title')['rating'].count())
ratings = ratings.rename(columns={"rating": "Number of Ratings"})
ratings['Mean Rating'] = user_movie_data.groupby('title')['rating'].mean()

ratings

---
## 5. Visualize Distributions

📖 **READ**: Visualizing the distribution of ratings helps us understand user behavior:
- **Number of Ratings distribution**: Are most movies rated by many users, or only a few?
- **Mean Rating distribution**: Do users tend to give high or low scores?
- **Scatter plot**: Is there a relationship between how popular a movie is and how well it's rated?

✏️ **TODO**: Plot the distribution of the **Number of Ratings** as a histogram.

*Hints:*
- *Use `plt.figure(figsize=(10, 5))` for a wider plot*
- *Use `ratings['Number of Ratings'].hist(bins=75)` to plot the histogram*
- *Add axis labels and a title*

In [ ]:
# Plot the Distribution of the Number of Ratings
plt.figure(figsize=(10, 5))
ratings['Number of Ratings'].hist(bins=75)
plt.xlabel("Number of Ratings")
plt.ylabel("Count")
plt.title("Distribution of Number of Ratings")
plt.show()

✏️ **TODO**: Plot the distribution of the **Mean Rating** as a histogram.

*Hints:*
- *Same approach as above, but use `ratings['Mean Rating'].hist(bins=75)`*

In [ ]:
# Plot the Distribution of the Average Rating Score
plt.figure(figsize=(10, 5))
ratings['Mean Rating'].hist(bins=75)
plt.xlabel("Average Rating")
plt.ylabel("Count")
plt.title("Distribution of Average Ratings")
plt.show()

✏️ **TODO**: Create a scatter plot showing the relationship between **Mean Rating** and **Number of Ratings**. Also compute and print the **correlation matrix** between the two variables.

*Hints:*
- *Use `print(ratings[['Mean Rating', 'Number of Ratings']].corr())` to print the full correlation matrix*
- *Use `plt.scatter(ratings['Mean Rating'], ratings['Number of Ratings'])` for the scatter plot*

In [ ]:
# Plot a Scatter Plot for the relation between Number of Ratings and Average Score
# and calculate correlation between both variables
print(ratings[['Mean Rating', 'Number of Ratings']].corr())
plt.scatter(ratings['Mean Rating'], ratings['Number of Ratings'])
plt.xlabel("Average Rating")
plt.ylabel("Number of Ratings")
plt.title("Relation between Number of Ratings and Average Rating")
plt.show()

### ✏️ TODO: Answer the following questions

**Q1: Looking at the histogram of Number of Ratings, how would you describe the distribution? Are most movies rated by many or few users?**

Your answer: The distribution is heavily **right-skewed** — the vast majority of movies have very few ratings, while only a handful of popular movies are rated by hundreds of users. This is typical of user-item interaction data and reflects the **long-tail** phenomenon in recommender systems.

**Q2: What does the correlation between Mean Rating and Number of Ratings tell us? Why might more popular movies have slightly higher average ratings?**

Your answer: There is a **moderate positive correlation** (~0.43). More popular movies (with more ratings) tend to have higher averages, likely because: (1) popular movies are often higher quality, (2) more ratings average out extreme scores (regression to the mean), and (3) users are more likely to rate movies they enjoyed (selection bias).

---
## 6. Build the User-Movie Matrix

📖 **READ**: To find movies similar to a given movie, we need to know how each user rated each movie. We create a **user-movie matrix** using a **pivot table**:

- **Rows** = Users
- **Columns** = Movies
- **Values** = Rating given by that user to that movie

Most cells will be `NaN` because users only rate a small fraction of all movies — this is the **sparsity** challenge of collaborative filtering.

**Important**: Do NOT remove movies with few ratings when building this matrix. We will filter later when looking at correlations.

✏️ **TODO**: Create the user-movie matrix using `pivot_table`. Display its shape and the matrix itself.

*Hints:*
- *Use `user_movie_data.pivot_table(index='user_id', columns='title', values='rating')`*
- *Print the shape to see how many users and movies there are*

In [ ]:
user_movie_matrix = user_movie_data.pivot_table(index='user_id', columns='title', values='rating')
user_movie_matrix

### ✏️ TODO: Answer the following question

**Q3: What does the shape of the user-movie matrix tell you? What do the `NaN` values represent? Why is this matrix so sparse?**

Your answer: The matrix has **users as rows** and **movies as columns**. `NaN` values indicate that a user **has not rated** that particular movie. The matrix is extremely sparse because each user typically rates only a small fraction of all available movies — a fundamental challenge for collaborative filtering.

In [ ]:
# What is the shape of this matrix? What does the shape mean?
user_movie_matrix.shape
# In total there are 944 users and a total of 1664 movies.
# If a value in a cell is NaN, this user has not evaluated this movie

---
## 7. Find Similar Movies

📖 **READ**: Now for the core of collaborative filtering! To find movies similar to a target movie, we:

1. Extract the column of ratings for the target movie
2. Compute the **Pearson correlation** between that column and every other movie's column using `corrwith()`
3. Movies with high correlation have **similar rating patterns** — users who liked the target movie also tended to like these movies

We drop `NaN` values from the correlation results because those mean no user rated both movies.

**Important**: We filter out movies with fewer than **75 ratings** to avoid spurious correlations from small sample sizes.

✏️ **TODO**: Find the most similar movies to **Star Wars (1977)** and **Liar Liar (1997)** based on rating patterns.

*Hints:*
- *Select a movie's ratings with `user_movie_matrix['Star Wars (1977)']`*
- *Compute correlations with `user_movie_matrix.corrwith(starwars_user_ratings)`*
- *Drop `NaN` values with `.dropna()` (NaN means no user rated both movies)*
- *Create a DataFrame from the correlation Series with column name `"Correlation"`*
- *Add the Number of Ratings using `.merge(ratings["Number of Ratings"], on="title")`*

In [ ]:
# Select the movies 'Star Wars (1977)' and 'Liar Liar (1997)'
starwars_user_ratings = user_movie_matrix['Star Wars (1977)']
liarliar_user_ratings = user_movie_matrix['Liar Liar (1997)']

# Dropna, because then no user has rated both movies
similarity_to_starwars = user_movie_matrix.corrwith(starwars_user_ratings).dropna()
similarity_to_liarliar = user_movie_matrix.corrwith(liarliar_user_ratings).dropna()

# Create DataFrame with correlation to other movies
correlation_to_starwars = pd.DataFrame(similarity_to_starwars, columns=["Correlation"])
correlation_to_liarliar = pd.DataFrame(similarity_to_liarliar, columns=["Correlation"])

# Add information on the Number of Ratings
correlation_to_starwars = correlation_to_starwars.merge(ratings["Number of Ratings"], on="title")
correlation_to_liarliar = correlation_to_liarliar.merge(ratings["Number of Ratings"], on="title")

✏️ **TODO**: Filter to keep only movies with **at least 75 ratings** and display the **top 3 most similar** movies to Star Wars.

*Hints:*
- *Filter: `correlation_to_starwars[correlation_to_starwars["Number of Ratings"] > 74]`*
- *Sort by `"Correlation"` in descending order with `.sort_values("Correlation", ascending=False)`*
- *Use `.head(3)` to show the top 3*

In [ ]:
# Only keep movies with at least 75 ratings and sort by correlation for Star Wars
correlation_to_starwars = correlation_to_starwars[correlation_to_starwars["Number of Ratings"] > 74].sort_values("Correlation", ascending=False)
correlation_to_starwars.head(3)

✏️ **TODO**: Do the same for **Liar Liar** — filter by 75+ ratings and show the top 3 most similar movies.

In [ ]:
# Only keep movies with at least 75 ratings and sort by correlation for Liar Liar
correlation_to_liarliar = correlation_to_liarliar[correlation_to_liarliar["Number of Ratings"] > 74].sort_values("Correlation", ascending=False)
correlation_to_liarliar.head(3)

### ✏️ TODO: Answer the following questions

**Q4: Do the similar movies to Star Wars make intuitive sense? What genre or characteristics do they share?**

Your answer: Yes — the movies most similar to Star Wars are *The Empire Strikes Back* and *Return of the Jedi*, i.e., the other films in the original **Star Wars trilogy**. This makes perfect sense because users who enjoy one Star Wars film tend to rate the entire trilogy similarly.

**Q5: Why do we filter out movies with fewer than 75 ratings? What would happen if we didn't?**

Your answer: Without filtering, movies with very few ratings would dominate the top results with **spuriously high correlations**. For example, if only 2 users rated both Star Wars and some obscure movie, and both gave them the same score, the correlation would be 1.0 — but this is statistically meaningless. The threshold of 75 ensures correlations are based on a **sufficient sample size**.

---
## Summary

In this lab, you learned how to:

| Section | Technique | Python Code |
| :--- | :--- | :--- |
| **1. Load Data** | Load CSV files with custom columns | `pd.read_csv(..., sep='\t', names=...)` |
| **2. Data Cleaning** | Extract & clean year from titles | `str[-6:]`, `str.replace(...)`, `.astype(int)` |
| **3. Merge** | Join DataFrames on a shared key | `pd.merge(..., on='item_id', how='left')` |
| **4. Rating Stats** | Group-by aggregation (count, mean) | `df.groupby('title')['rating'].count()` |
| **5. Visualization** | Histograms, scatter plots, correlation | `.hist(bins=75)`, `plt.scatter(...)`, `.corr()` |
| **6. User-Movie Matrix** | Pivot table for user-item ratings | `df.pivot_table(index=..., columns=..., values=...)` |
| **7. Similar Movies** | Collaborative filtering via correlation | `.corrwith(...)`, `.merge(...)`, filter by min ratings |

**Key takeaways:**
- **Collaborative filtering** finds similar items based on user behavior patterns, not item features
- A **user-movie matrix** is typically very sparse — most users rate only a few movies
- **Pearson correlation** between rating columns reveals movies with similar rating patterns
- Always **filter by minimum number of ratings** to avoid spurious correlations from small samples
- **Visualization** of rating distributions helps understand data quality and user behavior